# Lineage h5ad Structure Inspector
Quick inspection of each lineage file to identify the correct label column.

## Cell 1 — Imports + File Paths

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import scipy.sparse as sparse
from pathlib import Path

# ===== EDIT PATHS HERE =====
LINEAGE_FILES = {
    "epithelial": Path("/home/h2048/data/py/0317/epithelial_v2_7_HOTFIX/SELF/epithelial_scanvi_v2_7_HOTFIX_SELF_final.h5ad"),
    "tcell"     : Path("/home/h2048/data/py/0318/tnk_subcluster_retrain/adata_tnk_scanvi_ref_retrain_v1_2.h5ad"),
    "myeloid"   : Path("/home/h2048/data/py/0209/myeloid_validation_optimized/adata_myeloid_refined_FINAL.h5ad"),
    "bcell"     : Path("/home/h2048/data/py/0203/bcell_scarches_v4_1/results/scarches_packagebcell_reference_20260203.h5ad"),
    "stromal"   : Path("/home/h2048/data/py/0308/stromal_reintegration_v1_3/stromal_reintegrated_scvi_scanvi_v1_3.h5ad"),
}

# Keywords used to flag annotation-like columns for inspection
ANNOTATION_KEYWORDS = [
    "cell_type", "celltype", "scanvi", "leiden", "cluster",
    "ann_", "label", "annotation", "lineage", "subtype",
    "refined", "pred", "majority", "phase", "Cell_Type",
]

print("[OK] paths configured")
for k, p in LINEAGE_FILES.items():
    exists = "OK" if p.exists() else "MISSING"
    print(f"  [{exists}] {k}: {p.name}")

[OK] paths configured
  [OK] epithelial: epithelial_scanvi_v2_7_HOTFIX_SELF_final.h5ad
  [OK] tcell: adata_tnk_scanvi_ref_retrain_v1_2.h5ad
  [OK] myeloid: adata_myeloid_refined_FINAL.h5ad
  [MISSING] bcell: scarches_packagebcell_reference_20260203.h5ad
  [OK] stromal: stromal_reintegrated_scvi_scanvi_v1_3.h5ad


## Cell 2 — Inspector Function

In [2]:
def inspect_h5ad(name, path, top_n_cats=12, show_all_ann=True):
    """
    Load h5ad and print:
      - shape, .raw info, layer names, obsm keys
      - all annotation-like obs columns with value counts (top N)
      - raw type check for .raw.X and layers['counts']
    """
    print("=" * 72)
    print(f"  {name.upper()}  |  {path.name}")
    print("=" * 72)

    adata = sc.read_h5ad(path)

    # ── Basic shape ──────────────────────────────────────────────────────
    print(f"\nShape   : {adata.n_obs:,} cells x {adata.n_vars:,} genes")
    if adata.raw is not None:
        raw_X   = adata.raw.X
        raw_d   = raw_X.data if sparse.issparse(raw_X) else np.asarray(raw_X).ravel()
        _s      = raw_d[:min(50000, len(raw_d))]
        frac    = np.abs(_s - np.round(_s)).max() if len(_s) > 0 else 0
        raw_int = "integer-like" if frac < 1e-3 else f"NON-INTEGER (max_frac={frac:.4f})"
        print(f".raw    : {adata.raw.n_vars:,} genes  |  dtype={raw_X.dtype}  |  values={raw_int}")
    else:
        print(".raw    : None")

    if adata.layers:
        for lname, lX in adata.layers.items():
            ld   = lX.data if sparse.issparse(lX) else np.asarray(lX).ravel()
            _s   = ld[:min(50000, len(ld))]
            frac = np.abs(_s - np.round(_s)).max() if len(_s) > 0 else 0
            tag  = "integer-like" if frac < 1e-3 else f"non-integer(frac={frac:.3f})"
            fmt  = type(lX).__name__ if sparse.issparse(lX) else "ndarray"
            print(f"layer   : [{lname}]  {fmt}  dtype={lX.dtype}  {tag}")
    else:
        print("layers  : (none)")

    print(f"obsm    : {list(adata.obsm.keys())}")

    # ── Annotation columns ───────────────────────────────────────────────
    ann_cols = [
        col for col in adata.obs.columns
        if any(kw in col.lower() for kw in ANNOTATION_KEYWORDS)
        and col not in ("_scvi_batch","_scvi_labels",
                        "_scvi_extra_categorical_covs","_scvi_extra_continuous_covs")
    ]

    if show_all_ann:
        print(f"\n{'─'*72}")
        print(f"Annotation-like obs columns ({len(ann_cols)} found):")
        print(f"{'─'*72}")
        for col in ann_cols:
            series = adata.obs[col]
            dtype  = str(series.dtype)
            n_uniq = series.nunique()
            n_null = int(series.isna().sum())
            # top value counts
            try:
                vc = series.value_counts(dropna=False).head(top_n_cats)
                vc_str = "  |  ".join(f"{v}({n})" for v, n in vc.items())
            except Exception:
                vc_str = "(cannot compute)"
            null_tag = f"  [{n_null} null]" if n_null > 0 else ""
            print(f"\n  [{col}]  dtype={dtype}  unique={n_uniq}{null_tag}")
            print(f"  {vc_str}")

    # ── Non-annotation obs columns summary ───────────────────────────────
    other_cols = [c for c in adata.obs.columns if c not in ann_cols]
    print(f"\n{'─'*72}")
    print(f"Other obs columns ({len(other_cols)}): {other_cols}")

    del adata
    import gc; gc.collect()
    print()

print("[OK] inspect_h5ad defined")

[OK] inspect_h5ad defined


## Epithelial

In [3]:
inspect_h5ad("epithelial", LINEAGE_FILES["epithelial"])

  EPITHELIAL  |  epithelial_scanvi_v2_7_HOTFIX_SELF_final.h5ad

Shape   : 278,380 cells x 4,024 genes
.raw    : 53,923 genes  |  dtype=float64  |  values=NON-INTEGER (max_frac=0.5000)
layer   : [counts]  csr_matrix  dtype=float32  integer-like
layer   : [log1p]  csc_matrix  dtype=float64  non-integer(frac=0.500)
layer   : [raw_counts]  csc_matrix  dtype=float64  non-integer(frac=0.500)
obsm    : ['X_pca', 'X_pca_harmony', 'X_scanvi_fine', 'X_scanvi_major', 'X_scvi', 'X_umap', 'X_umap_fine', 'X_umap_harmony', 'X_umap_major', '_scvi_extra_continuous_covs']

────────────────────────────────────────────────────────────────────────
Annotation-like obs columns (83 found):
────────────────────────────────────────────────────────────────────────

  [cellType]  dtype=category  unique=8
  Unknown(277244)  |  basal(585)  |  unk_epi(519)  |  goblet+club(12)  |  unk_neut(8)  |  hillock(8)  |  ciliated(3)  |  prolifT(1)

  [decontX_clusters]  dtype=category  unique=233
  Unknown(208864)  |  NP32-NB-

## Tcell

In [4]:
inspect_h5ad("tcell", LINEAGE_FILES["tcell"])

  TCELL  |  adata_tnk_scanvi_ref_retrain_v1_2.h5ad

Shape   : 38,904 cells x 4,000 genes
.raw    : 32,723 genes  |  dtype=float32  |  values=NON-INTEGER (max_frac=0.4965)
layer   : [counts]  csr_matrix  dtype=float32  integer-like
layer   : [log1p]  csr_matrix  dtype=float32  non-integer(frac=0.500)
layer   : [raw_counts]  csr_matrix  dtype=float64  integer-like
obsm    : ['X_scanvi', 'X_scanvi_refined', 'X_scvi', 'X_umap', 'X_umap_refined', '_scvi_extra_categorical_covs']

────────────────────────────────────────────────────────────────────────
Annotation-like obs columns (98 found):
────────────────────────────────────────────────────────────────────────

  [cellType]  dtype=category  unique=4
  Unknown(38827)  |  CD4_T(55)  |  CD8_T(12)  |  unk_neut(10)

  [decontX_clusters]  dtype=category  unique=32
  Unknown(35472)  |  NP17-NB-2(644)  |  NP13-NB-2(405)  |  NP19-NB-3(397)  |  NP10-NB-2(370)  |  NP16-NB-4(370)  |  NP20-NB-2(294)  |  NP28-NB-2(155)  |  NP44-NB_v1.0-5(112)  |  AN9-NB

## Myeloid

In [5]:
inspect_h5ad("myeloid", LINEAGE_FILES["myeloid"])

  MYELOID  |  adata_myeloid_refined_FINAL.h5ad

Shape   : 54,731 cells x 35,112 genes
.raw    : 35,112 genes  |  dtype=float64  |  values=integer-like
layer   : [counts]  csr_matrix  dtype=float64  integer-like
layer   : [log1p]  csr_matrix  dtype=float64  non-integer(frac=0.497)
layer   : [raw_counts]  csr_matrix  dtype=float64  integer-like
obsm    : ['X_cnmf_usages', 'X_pca', 'X_scanvi', 'X_scvi', 'X_umap', 'X_umap_bbknn', 'X_umap_scanvi', 'X_umap_scvi', '_scvi_extra_categorical_covs', 'scanvi_probabilities']

────────────────────────────────────────────────────────────────────────
Annotation-like obs columns (83 found):
────────────────────────────────────────────────────────────────────────

  [cellType]  dtype=category  unique=13
  Unknown(50935)  |  unk_neut(2057)  |  G5c_aged(845)  |  G5a_naive(670)  |  M2-mac(72)  |  G5c_naive(69)  |  M1-mac(25)  |  pDC(24)  |  G5b(20)  |  unk_mac(10)  |  unk_T(2)  |  IFNexp-mac(1)

  [decontX_clusters]  dtype=category  unique=59
  Unknown(501

## Bcell

In [7]:
inspect_h5ad("bcell", LINEAGE_FILES["bcell"])

  BCELL  |  scarches_packagebcell_reference_20260203.h5ad


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/home/h2048/data/py/0203/bcell_scarches_v4_1/results/scarches_packagebcell_reference_20260203.h5ad', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

## Stromal

In [8]:
inspect_h5ad("stromal", LINEAGE_FILES["stromal"])

  STROMAL  |  stromal_reintegrated_scvi_scanvi_v1_3.h5ad

Shape   : 56,689 cells x 4,000 genes
.raw    : 36,789 genes  |  dtype=float64  |  values=NON-INTEGER (max_frac=0.4873)
layer   : [counts]  csr_matrix  dtype=float64  non-integer(frac=0.500)
layer   : [log1p]  csr_matrix  dtype=float64  non-integer(frac=0.500)
obsm    : ['X_pca', 'X_scanvi', 'X_scvi', 'X_umap', 'X_umap_bbknn', 'X_umap_scanvi', 'X_umap_scvi']

────────────────────────────────────────────────────────────────────────
Annotation-like obs columns (86 found):
────────────────────────────────────────────────────────────────────────

  [cellType]  dtype=category  unique=1
  Unknown(56689)

  [decontX_clusters]  dtype=category  unique=2
  Unknown(56677)  |  NP19-NB-6(12)

  [cell_type_ontology_term_id]  dtype=category  unique=2
  Unknown(56677)  |  CL:0000442(12)

  [cell_type]  dtype=category  unique=3
  Endothelial(32657)  |  Fibroblast(19279)  |  SMC(4753)

  [CellType]  dtype=category  unique=1
  Unknown(56689)

  [Br

## Summary — Candidate Label Columns

In [ ]:
print("="*72)
print("CANDIDATE LABEL COLUMN SUMMARY")
print("="*72)
print("Fill in the 'chosen' column based on inspection above.")
print()

rows = []
for name, path in LINEAGE_FILES.items():
    adata = sc.read_h5ad(path)
    ann_cols = [
        col for col in adata.obs.columns
        if any(kw in col.lower() for kw in ANNOTATION_KEYWORDS)
        and col not in ("_scvi_batch","_scvi_labels",
                        "_scvi_extra_categorical_covs","_scvi_extra_continuous_covs")
    ]
    for col in ann_cols:
        n_uniq = adata.obs[col].nunique()
        n_null = int(adata.obs[col].isna().sum())
        rows.append({"lineage": name, "column": col,
                     "unique": n_uniq, "null": n_null})
    del adata
    import gc; gc.collect()

df = pd.DataFrame(rows)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 60)
print(df.to_string(index=False))